# MediBridge AI — Kaggle Demo
Run each cell top-to-bottom. The last cell prints a public cloudflare URL — open it in your browser.

In [ ]:
# ── 1. Clone / update repo ────────────────────────────────────────────────
import os

REPO = "https://github.com/tripathiji1312/Medi-BridgeAI.git"
if not os.path.exists("/kaggle/working/Medi-BridgeAI"):
    !git clone --depth 1 {REPO} /kaggle/working/Medi-BridgeAI
else:
    !git -C /kaggle/working/Medi-BridgeAI pull --ff-only

os.chdir("/kaggle/working/Medi-BridgeAI")
print("repo ready")

In [ ]:
# ── 2. Install Python deps for all services ────────────────────────────────
# Kaggle already has torch — skip the 2GB re-download.
!pip install -q uvicorn fastapi websockets faster-whisper \
    transformers sentencepiece speechbrain \
    python-multipart httpx pydantic pydantic-settings \
    sentence-transformers presidio-analyzer presidio-anonymizer mediapipe
print("deps installed")

In [ ]:
# ── 3. Install Node 20 (Kaggle ships an old Node that can't run TypeScript) ─
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node --version && npm --version

In [ ]:
# ── 4. Install JS deps & build frontend ───────────────────────────────────
os.chdir("/kaggle/working/Medi-BridgeAI")
!npm install --silent
# Build the frontend pointing at the kaggle cloudflare URL — we'll fill in
# the real URL after cloudflared starts (see cell 7). For now build with
# a placeholder; we'll patch it below once the tunnel URL is known.
print("JS deps installed — frontend will be built in cell 7 after URL is known")

In [ ]:
# ── 5. Set HF token from Kaggle secrets (optional) ──────────────────────
# Add your HuggingFace token in Kaggle → Add-ons → Secrets with key: HF_TOKEN (if using private models)
import os

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["MEDIBRIDGE_FIXTURE_MODE"] = ""   # empty = real models
os.environ["ASR_MODEL_SIZE"] = "small"
os.environ["ASR_DEVICE"] = "cuda"
os.environ["ASR_COMPUTE_TYPE"] = "float16"
os.environ["ASR_RMS_THRESHOLD"] = "250"
os.environ["ASR_LANGUAGE"] = "auto"
os.environ["MT_MODEL_NAME"] = "facebook/nllb-200-distilled-600M"
os.environ["DIARIZATION_MODEL_SOURCE"] = "speechbrain/spkrec-ecapa-voxceleb"
os.environ["CLINICAL_NLP_URL"] = "http://localhost:8002"
os.environ["ORCHESTRATOR_URL"] = "http://localhost:8004"
os.environ["VISION_SERVICE_URL"] = "http://localhost:8003"
os.environ["SPEECH_PIPELINE_WS_URL"] = "ws://localhost:8001/ws/transcribe"
os.environ["JWT_SECRET"] = "demo-only"
print("env set (GPU enabled, VAD threshold=250, auto-lang)")

In [ ]:
# ── 6. Pre-warm models (downloads to ~/.cache on first run) ───────────────
# Runs in separate sub-processes to isolate packages and release GPU VRAM before starting services.
import subprocess, sys, os

BASE = "/kaggle/working/Medi-BridgeAI"

print("── Pre-warming Speech Pipeline Models (ASR, MT, TTS, Diarization) ──")
p1 = subprocess.run([sys.executable, "-c", 'import os, sys\nsys.path.insert(0, ".")\nfrom app.asr.provider_factory import get_asr_provider\nfrom app.mt.provider_factory import get_mt_provider\nfrom app.tts.provider_factory import get_tts_provider\nfrom app.diarization.provider_factory import get_embedding_provider\n\nprint("loading ASR (GPU)...")\nget_asr_provider()\nprint("loading MT (GPU)...")\nget_mt_provider()\nprint("loading TTS Hindi & English (GPU)...")\ntts = get_tts_provider()\nif hasattr(tts, "prewarm"):\n    tts.prewarm(["en", "hi"])\nprint("loading diarization...")\nget_embedding_provider()\nprint("speech-pipeline models ready ✓")\n'], cwd=f"{BASE}/services/speech-pipeline", env=os.environ)
if p1.returncode != 0:
    print("Warning: speech-pipeline pre-warming encountered an issue (will load on demand)")

print("\n── Pre-warming Clinical NLP Models (Biomedical NER, Clinical Summarizer) ──")
p2 = subprocess.run([sys.executable, "-c", 'import os, sys\nsys.path.insert(0, ".")\ntry:\n    from app.ner.neural_ner_provider import get_neural_ner_provider\n    print("loading Biomedical NER (GPU)...")\n    get_neural_ner_provider()._ensure_loaded()\n    print("Clinical NER ready ✓")\nexcept Exception as e:\n    print(f"Clinical NER init deferred: {e}")\n'], cwd=f"{BASE}/services/clinical-nlp", env=os.environ)
if p2.returncode != 0:
    print("Warning: clinical-nlp pre-warming encountered an issue (will load on demand)")

print("\n✅ All models ready on GPU ✓")

In [ ]:
# ── 7. Install cloudflared & start TWO tunnels ───────────────────────────
# Tunnel A → port 5173 (frontend static files, this is the URL you open)
# Tunnel B → port 4000 (gateway API + WebSocket)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, threading, time, re

tunnel_url   = {"value": None}   # frontend URL (open this in browser)
gateway_url  = {"value": None}   # gateway URL (baked into frontend build)

def start_tunnel(port, store):
    proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
    )
    for line in proc.stderr:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            store["value"] = m.group(0)
            print(f"Tunnel :{port} → {store['value']}")

threading.Thread(target=start_tunnel, args=(5173, tunnel_url),  daemon=True).start()
time.sleep(2)
threading.Thread(target=start_tunnel, args=(4000, gateway_url), daemon=True).start()

# Wait up to 40s for both
for _ in range(40):
    time.sleep(1)
    if tunnel_url["value"] and gateway_url["value"]:
        break

if not tunnel_url["value"] or not gateway_url["value"]:
    raise RuntimeError(f"Tunnels not ready — frontend={tunnel_url['value']} gateway={gateway_url['value']}")

print(f"\n✅ Open this in your browser: {tunnel_url['value']}")

In [ ]:
# ── 8. Build frontend pointing at the gateway tunnel ─────────────────────
import os
gw = gateway_url["value"]
os.chdir("/kaggle/working/Medi-BridgeAI/apps/web")
os.environ["VITE_GATEWAY_HTTP_URL"] = gw
os.environ["VITE_GATEWAY_WS_URL"]   = gw.replace("https://", "wss://") + "/ws/transcribe"

!npm run build
print("frontend built")

In [ ]:
# ── 9. Launch all backend services ────────────────────────────────────────
import subprocess, os

BASE = "/kaggle/working/Medi-BridgeAI"
PYTHON = sys.executable

def start(name, cwd, cmd, env_extra=None):
    env = {**os.environ, **(env_extra or {})}
    log = open(f"/kaggle/working/{name}.log", "w")
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=log, stderr=log)
    print(f"{name} started (pid {p.pid})")
    return p

procs = []
procs.append(start("speech-pipeline", f"{BASE}/services/speech-pipeline",
    [PYTHON, "-m", "uvicorn", "app.main:app", "--port", "8001"],
    {"PYTHONPATH": ".", "MEDIBRIDGE_FIXTURE_MODE": "",
     "ASR_MODEL_SIZE": "small",
     "ASR_DEVICE": "cuda",
     "ASR_COMPUTE_TYPE": "float16",
     "ASR_RMS_THRESHOLD": "250",
     "ASR_LANGUAGE": "auto",
     "CLINICAL_NLP_URL": "http://localhost:8002",
     "ORCHESTRATOR_URL": "http://localhost:8004"}))

procs.append(start("clinical-nlp", f"{BASE}/services/clinical-nlp",
    [PYTHON, "-m", "uvicorn", "app.main:app", "--port", "8002"],
    {"PYTHONPATH": ".", "MEDIBRIDGE_FIXTURE_MODE": ""}))

procs.append(start("vision-service", f"{BASE}/services/vision-service",
    [PYTHON, "-m", "uvicorn", "app.main:app", "--port", "8003"],
    {"PYTHONPATH": ".", "MEDIBRIDGE_FIXTURE_MODE": ""}))

procs.append(start("orchestrator", f"{BASE}/services/orchestrator",
    [PYTHON, "-m", "uvicorn", "app.main:app", "--port", "8004"],
    {"PYTHONPATH": "."}))

import time; time.sleep(5)
print("backend services up")

In [ ]:
# ── 10. Start gateway + static file server (serves built frontend) ────────
import subprocess, sys, time, urllib.request, os

BASE = "/kaggle/working/Medi-BridgeAI"

gw_env = {
    **os.environ,
    "SPEECH_PIPELINE_WS_URL": "ws://localhost:8001/ws/transcribe",
    "ORCHESTRATOR_URL": "http://localhost:8004",
    "VISION_SERVICE_URL": "http://localhost:8003",
    "GATEWAY_PORT": "4000",
    "JWT_SECRET": "demo-only",
}
gw_log = open("/kaggle/working/gateway.log", "w")
# tsx lives in the root node_modules (npm workspaces)
gw = subprocess.Popen(
    [f"{BASE}/node_modules/.bin/tsx", "src/server.ts"],
    cwd=f"{BASE}/services/gateway",
    env=gw_env, stdout=gw_log, stderr=gw_log
)
print(f"gateway pid {gw.pid} — waiting for it to bind...")

for _ in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:4000/health", timeout=2)
        print("gateway is up ✓")
        break
    except Exception:
        pass
else:
    import subprocess as sp
    sp.run(["tail", "-30", "/kaggle/working/gateway.log"])
    raise RuntimeError("Gateway never came up — see log above")

# Serve the built frontend on 5173 (cloudflared tunnels this port)
import http.server, threading
os.chdir(f"{BASE}/apps/web/dist")

class SPAHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *a): pass
    def do_GET(self):
        if not os.path.exists(self.directory + self.path.split('?')[0]):
            self.path = '/index.html'
        super().do_GET()

server = http.server.HTTPServer(('0.0.0.0', 5173), SPAHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("frontend server started on :5173")

print(f"\n✅ Open this URL in your browser:\n   {tunnel_url['value']}\n")
print(f"   (gateway at: {gateway_url['value']})")

In [ ]:
# ── (optional) Check logs if something is broken ─────────────────────────
service = "speech-pipeline"   # change to: gateway, clinical-nlp, orchestrator, vision-service
!tail -30 /kaggle/working/{service}.log